# CHAKRA P1: waste item classifier

Attach these two Kaggle datasets to this notebook before running it:

1. [Garbage Classification (12 classes) ENHANCED](https://www.kaggle.com/datasets/huberthamelin/garbage-classification-labels-corrections), using only its `garbage_classification_enhanced` folder.
2. [Garbage Classification](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification), the six-class clean-image source.

Set **Accelerator = GPU** and **Internet = On**. Kaggle may require phone verification to unlock those settings. The notebook downloads [RealWaste from UCI](https://archive.ics.uci.edu/dataset/908/realwaste) directly and holds all of it out for the domain-shift test. UCI currently lists RealWaste as **CC BY 4.0**; check licenses for all sources before commercial use.

The model has six internal outputs in the CHAKRA checkpoint order: battery, metal, organic, other, paper, plastic. Battery is a safety flag, never an "other" disposal item. The model files are saved to `/kaggle/working`.


In [ ]:
from pathlib import Path
import os

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
print('CUDA:', __import__('torch').cuda.is_available())
if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle notebook settings before training.')

for root, dirs, files in os.walk(INPUT):
    depth = Path(root).relative_to(INPUT).parts
    if len(depth) > 3:
        dirs[:] = []
        continue
    print(root, len(files), 'files')

def folders_named(name):
    return sorted(p for p in INPUT.rglob('*') if p.is_dir() and p.name == name)

enhanced = folders_named('garbage_classification_enhanced')
if len(enhanced) != 1:
    raise RuntimeError(f'Expected one enhanced folder; found {enhanced}. Attach the 12-class Kaggle dataset.')
KAGGLE12_DIR = enhanced[0]

six_labels = {'cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'}
clean_roots = []
for folder in INPUT.rglob('*'):
    if not folder.is_dir() or KAGGLE12_DIR in folder.parents or folder == KAGGLE12_DIR:
        continue
    children = {p.name.lower().strip() for p in folder.iterdir() if p.is_dir()}
    if children == six_labels:
        clean_roots.append(folder)
if not clean_roots:
    raise RuntimeError('No six-class folder found. Attach the six-class Kaggle dataset.')
clean_roots.sort()
TRASHNET_DIR = clean_roots[0]
if len(clean_roots) > 1:
    print('Several six-class copies found; using one:', TRASHNET_DIR)
print('Enhanced:', KAGGLE12_DIR)
print('Six-class:', TRASHNET_DIR)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

UCI_ZIP_URL = 'https://archive.ics.uci.edu/static/public/908/realwaste.zip'
REALWASTE_DIR = WORK / 'realwaste-main' / 'RealWaste'
archive = WORK / 'realwaste.zip'
if not REALWASTE_DIR.is_dir():
    if not archive.exists():
        print('Downloading official UCI RealWaste archive (about 657 MB)...', flush=True)
        with urlopen(UCI_ZIP_URL, timeout=120) as response, archive.open('wb') as target:
            while block := response.read(1024 * 1024):
                target.write(block)
        print('Downloaded:', archive.stat().st_size, 'bytes')
    with ZipFile(archive) as zipped:
        for info in zipped.infolist():
            destination = (WORK / info.filename).resolve()
            if not destination.is_relative_to(WORK.resolve()):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
        zipped.extractall(WORK)

expected = {
    'Cardboard': 461, 'Food Organics': 411, 'Glass': 420,
    'Metal': 790, 'Miscellaneous Trash': 495, 'Paper': 500,
    'Plastic': 921, 'Textile Trash': 318, 'Vegetation': 436,
}
actual = {name: len(list((REALWASTE_DIR / name).glob('*.jpg'))) for name in expected}
print('RealWaste:', REALWASTE_DIR, actual)
if actual != expected:
    raise ValueError(f'RealWaste does not match the UCI class counts: {actual}')


In [ ]:
%%writefile /kaggle/working/prepare_dataset.py
"""Build a source-aware image manifest for CHAKRA's item classifier.

RealWaste is a held-out domain test. Exact duplicate images found in that
source are removed from training sources, regardless of source argument order.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
from collections import Counter, defaultdict
from pathlib import Path


CLASSES = ("battery", "metal", "organic", "other", "paper", "plastic")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
DEFAULT_MAP = {
    "paper": "paper",
    "cardboard": "paper",
    "plastic": "plastic",
    "metal": "metal",
    "biological": "organic",
    "food organics": "organic",
    "vegetation": "organic",
    "trash": "other",
    "misc trash": "other",
    "miscellaneous trash": "other",
    "glass": "other",
    "green glass": "other",
    "brown glass": "other",
    "white glass": "other",
    "clothes": "other",
    "shoes": "other",
    "textile trash": "other",
    "battery": "battery",
    "batteries": "battery",
}
SOURCE_NAMES = {"kaggle12", "realwaste", "trashnet"}


def normalize_label(value: str) -> str:
    return "_".join(value.strip().lower().replace("-", " ").replace("_", " ").split())


def parse_pair(value: str, *, kind: str) -> tuple[str, str]:
    if "=" not in value:
        raise ValueError(f"{kind} must be NAME=VALUE: {value!r}")
    name, target = value.split("=", 1)
    if not name.strip() or not target.strip():
        raise ValueError(f"{kind} must be NAME=VALUE: {value!r}")
    return name.strip(), target.strip()


def image_folders(root: Path):
    if not root.is_dir():
        raise ValueError(f"Dataset directory does not exist: {root}")
    for folder in sorted(p for p in root.rglob("*") if p.is_dir()):
        images = sorted(
            (p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES),
            key=lambda p: str(p).lower(),
        )
        if images:
            yield folder, images
    root_images = sorted(
        (p for p in root.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES),
        key=lambda p: str(p).lower(),
    )
    if root_images:
        yield root, root_images


def file_md5(path: Path) -> str:
    digest = hashlib.md5()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare(sources: dict[str, Path], label_map: dict[str, str], output: Path) -> dict:
    if not sources:
        raise ValueError("Provide at least one --source NAME=PATH")
    candidates = []
    unknown = defaultdict(int)
    for source, root in sources.items():
        if source not in SOURCE_NAMES:
            raise ValueError(f"Unknown source {source!r}; use {', '.join(sorted(SOURCE_NAMES))}")
        if source == "kaggle12" and (root / "garbage_classification_corrected").is_dir() and (
            root / "garbage_classification_enhanced"
        ).is_dir():
            raise ValueError(
                "Point kaggle12 at garbage_classification_enhanced, not the parent folder "
                "that also contains garbage_classification_corrected"
            )
        for folder, images in image_folders(root):
            original = folder.name
            target = label_map.get(normalize_label(original))
            if target is None:
                unknown[(source, original)] += len(images)
                continue
            for path in images:
                candidates.append((source, original, target, path))
    if unknown:
        lines = ["Unmapped label folders (add --map FOLDER=CLASS; no manifest written):"]
        lines.extend(
            f"  {source}/{label}: {count} images"
            for (source, label), count in sorted(unknown.items())
        )
        raise ValueError("\n".join(lines))

    # Hash before splitting, and retain RealWaste when a byte-identical image
    # appears in both training and holdout sources.
    hashed = [(source, original, target, path, file_md5(path)) for source, original, target, path in candidates]
    hashed.sort(key=lambda row: (row[0] != "realwaste", row[0], str(row[3]).lower()))
    seen = set()
    kept = []
    dropped = Counter()
    for source, original, target, path, md5 in hashed:
        if md5 in seen:
            dropped[source] += 1
            continue
        seen.add(md5)
        kept.append((str(path.resolve()), source, original, target, md5))
    kept.sort(key=lambda row: (row[1], row[2].lower(), row[0].lower()))
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(("path", "source", "original_label", "target_class", "md5"))
        writer.writerows(kept)
    counts = Counter((source, target) for _, source, _, target, _ in kept)
    print(f"Manifest: {output} ({len(kept)} images; {sum(dropped.values())} duplicate copies removed)")
    for source in sorted(sources):
        print(f"{source}: " + ", ".join(f"{label}={counts[source, label]}" for label in CLASSES))
    return {"n_images": len(kept), "counts": counts, "duplicates_removed": dict(dropped)}


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--source", action="append", default=[], metavar="NAME=PATH")
    parser.add_argument("--map", action="append", default=[], metavar="FOLDER=CLASS")
    parser.add_argument("--out", required=True, type=Path, metavar="CSV")
    args = parser.parse_args()
    try:
        sources = {}
        for entry in args.source:
            name, path = parse_pair(entry, kind="source")
            if name in sources:
                raise ValueError(f"Duplicate source name: {name}")
            sources[name] = Path(path)
        label_map = {normalize_label(name): target for name, target in DEFAULT_MAP.items()}
        for entry in args.map:
            folder, target = parse_pair(entry, kind="map")
            target = normalize_label(target)
            if target not in CLASSES:
                raise ValueError(f"Unknown target class {target!r}; use {', '.join(CLASSES)}")
            label_map[normalize_label(folder)] = target
        prepare(sources, label_map, args.out)
    except (OSError, ValueError) as exc:
        parser.exit(2, f"{exc}\n")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/train_classifier.py
"""Fine-tune a six-output MobileNetV3 Small for five bins plus battery safety."""

from __future__ import annotations

import argparse
import csv
import hashlib
import random
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from prepare_dataset import CLASSES


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def read_manifest(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    required = {"path", "source", "original_label", "target_class", "md5"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError(f"Manifest has no rows or misses columns: {sorted(required)}")
    unknown = sorted({row["target_class"] for row in rows} - set(CLASSES))
    if unknown:
        raise ValueError(f"Unknown target classes: {unknown}")
    if len({row["md5"] for row in rows}) != len(rows):
        raise ValueError("Manifest still contains duplicate image hashes")
    return rows


def manifest_fingerprint(rows: list[dict[str, str]]) -> str:
    canonical = sorted((r["source"], r["target_class"], r["md5"]) for r in rows)
    content = "\n".join("\t".join(row) for row in canonical).encode("utf-8")
    return hashlib.sha256(content).hexdigest()


def split_records(rows: list[dict[str, str]], holdout_source: str = "realwaste", seed: int = 0):
    """Return deterministic, class-stratified train/val/test/shift partitions."""
    from sklearn.model_selection import train_test_split

    trainable = [row for row in rows if row["source"] != holdout_source]
    shift = [row for row in rows if row["source"] == holdout_source]
    if not shift:
        raise ValueError(f"Holdout source {holdout_source!r} is absent from the manifest")
    counts = Counter(row["target_class"] for row in trainable)
    sparse = {name: counts[name] for name in CLASSES if counts[name] < 10}
    if sparse:
        raise ValueError(f"At least 10 training-source images per class are needed: {sparse}")
    labels = [row["target_class"] for row in trainable]
    train, remainder = train_test_split(
        trainable, test_size=0.2, random_state=seed, stratify=labels
    )
    val, test = train_test_split(
        remainder,
        test_size=0.5,
        random_state=seed,
        stratify=[row["target_class"] for row in remainder],
    )
    return train, val, test, shift


def make_transforms(train: bool):
    from torchvision import transforms

    if train:
        steps = [
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2),
        ]
    else:
        steps = [transforms.Resize(256), transforms.CenterCrop(224)]
    return transforms.Compose(
        steps + [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    )


class ImageRows:
    def __init__(self, rows: list[dict[str, str]], transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        from PIL import Image

        row = self.rows[index]
        with Image.open(row["path"]) as raw:
            image = raw.convert("RGB")
        return self.transform(image), CLASSES.index(row["target_class"])


def evaluate_accuracy(model, loader, device) -> float:
    import torch

    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(device))
            correct += int((logits.argmax(1).cpu() == labels).sum())
            total += len(labels)
    return correct / total if total else 0.0


def train(args) -> Path:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

    rows = read_manifest(args.manifest)
    train_rows, val_rows, test_rows, shift_rows = split_records(rows, args.holdout_source, args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"CUDA available: {torch.cuda.is_available()} | device: {device}")
    print(
        f"Split: train={len(train_rows)}, val={len(val_rows)}, "
        f"random_test={len(test_rows)}, shift_test={len(shift_rows)}"
    )
    counts = Counter(row["target_class"] for row in train_rows)
    weights = [1.0 / counts[label] for label in CLASSES]
    mean_weight = sum(weights) / len(weights)
    weights = torch.tensor([weight / mean_weight for weight in weights], dtype=torch.float32, device=device)
    train_loader = DataLoader(
        ImageRows(train_rows, make_transforms(True)),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
    )
    val_loader = DataLoader(
        ImageRows(val_rows, make_transforms(False)),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
    )
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES))
    model.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    best_accuracy = -1.0
    best_epoch = 0
    best_state = None
    for epoch in range(1, args.epochs + 1):
        started = time.perf_counter()
        model.train()
        running_loss = 0.0
        n_seen = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += float(loss) * len(labels)
            n_seen += len(labels)
        val_accuracy = evaluate_accuracy(model, val_loader, device)
        elapsed = time.perf_counter() - started
        print(
            f"epoch {epoch}/{args.epochs}: train_loss={running_loss / n_seen:.4f} "
            f"val_accuracy={val_accuracy:.4f} seconds={elapsed:.1f}",
            flush=True,
        )
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    assert best_state is not None
    args.out_dir.mkdir(parents=True, exist_ok=True)
    output = args.out_dir / "item_classifier.pt"
    torch.save(
        {
            "state_dict": best_state,
            "classes": list(CLASSES),
            "arch": "mobilenet_v3_small",
            "img_size": 224,
            "mean": list(IMAGENET_MEAN),
            "std": list(IMAGENET_STD),
            "model_version": "mnv3s-" + datetime.now(timezone.utc).strftime("%Y%m%d"),
        },
        output,
    )
    print(f"Saved {output} (best epoch {best_epoch}, validation accuracy {best_accuracy:.4f})")
    return output


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", required=True, type=Path)
    parser.add_argument("--holdout-source", default="realwaste")
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--out-dir", type=Path, default=Path("/kaggle/working"))
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--num-workers", type=int, default=2)
    args = parser.parse_args()
    if args.epochs < 1 or args.batch_size < 1 or args.num_workers < 0 or args.lr <= 0:
        parser.error("epochs, batch-size and lr must be positive; num-workers must be nonnegative")
    train(args)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/eval_classifier.py
"""Evaluate CHAKRA's classifier on random, source-shift, and campus photos."""

from __future__ import annotations

import argparse
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from prepare_dataset import CLASSES, IMAGE_SUFFIXES, normalize_label
from train_classifier import (
    ImageRows,
    make_transforms,
    read_manifest,
    split_records,
)


def campus_rows(directory: Path) -> list[dict[str, str]]:
    if not directory.is_dir():
        raise ValueError(f"Campus photo directory does not exist: {directory}")
    rows = []
    for folder in sorted(p for p in directory.iterdir() if p.is_dir()):
        label = normalize_label(folder.name)
        if label not in CLASSES:
            raise ValueError(f"Unknown campus folder {folder.name!r}; use one of {', '.join(CLASSES)}")
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                rows.append({"path": str(path), "target_class": label, "source": "campus"})
    if not rows:
        raise ValueError(f"No labeled campus images found in {directory}")
    return rows


def evaluate_set(model, rows, device, batch_size: int) -> dict:
    import torch
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    from torch.utils.data import DataLoader

    loader = DataLoader(ImageRows(rows, make_transforms(False)), batch_size=batch_size, shuffle=False)
    actual = []
    predicted = []
    model.eval()
    with torch.inference_mode():
        for images, labels in loader:
            output = model(images.to(device))
            predicted.extend(output.argmax(1).cpu().tolist())
            actual.extend(labels.tolist())
    indices = list(range(len(CLASSES)))
    counts = Counter(actual)
    return {
        "n_images": len(rows),
        "accuracy": float(accuracy_score(actual, predicted)),
        "per_class": classification_report(
            actual,
            predicted,
            labels=indices,
            target_names=list(CLASSES),
            output_dict=True,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(actual, predicted, labels=indices).tolist(),
        "label_order": list(CLASSES),
        "absent_classes": [CLASSES[index] for index in indices if counts[index] == 0],
    }


def evaluate(args) -> dict:
    import torch
    from torch import nn
    from torchvision.models import mobilenet_v3_small

    rows = read_manifest(args.manifest)
    checkpoint = torch.load(args.model, map_location="cpu", weights_only=True)
    if checkpoint.get("classes") != list(CLASSES):
        raise ValueError("Checkpoint class order differs from the six-class CHAKRA contract")
    holdout_source = args.holdout_source
    _, _, random_test, shift_test = split_records(rows, holdout_source, args.seed)
    model = mobilenet_v3_small(weights=None)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES))
    model.load_state_dict(checkpoint["state_dict"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    report = {
        "model_version": checkpoint["model_version"],
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "classes": list(CLASSES),
        "holdout_source": holdout_source,
        "random_test": evaluate_set(model, random_test, device, args.batch_size),
        "shift_test": evaluate_set(model, shift_test, device, args.batch_size),
    }
    if args.campus_dir:
        report["campus_test"] = evaluate_set(model, campus_rows(args.campus_dir), device, args.batch_size)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    for name in ("random_test", "shift_test", "campus_test"):
        if name in report:
            metrics = report[name]
            print(f"{name}: n={metrics['n_images']} accuracy={metrics['accuracy']:.4f} absent={metrics['absent_classes']}")
    print(f"Saved {args.out}")
    return report


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", required=True, type=Path)
    parser.add_argument("--manifest", required=True, type=Path)
    parser.add_argument("--holdout-source", default="realwaste")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--campus-dir", type=Path)
    parser.add_argument("--out", type=Path, default=Path("classifier_report.json"))
    parser.add_argument("--batch-size", type=int, default=64)
    args = parser.parse_args()
    evaluate(args)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
import subprocess
import sys

def run(*args):
    print('Running:', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, *map(str, args)], check=True)

run(WORK / 'prepare_dataset.py',
    '--source', f'kaggle12={KAGGLE12_DIR}',
    '--source', f'trashnet={TRASHNET_DIR}',
    '--source', f'realwaste={REALWASTE_DIR}',
    '--out', WORK / 'manifest.csv')

run(WORK / 'train_classifier.py',
    '--manifest', WORK / 'manifest.csv',
    '--holdout-source', 'realwaste',
    '--epochs', '8', '--batch-size', '64', '--lr', '0.0003',
    '--out-dir', WORK, '--seed', '0', '--num-workers', '2')

run(WORK / 'eval_classifier.py',
    '--model', WORK / 'item_classifier.pt',
    '--manifest', WORK / 'manifest.csv',
    '--holdout-source', 'realwaste',
    '--out', WORK / 'classifier_report.json')


In [ ]:
import json

report = json.loads((WORK / 'classifier_report.json').read_text())
for name in ('random_test', 'shift_test'):
    result = report[name]
    print(name, 'images =', result['n_images'], 'accuracy =', round(result['accuracy'], 4))
    print('absent classes:', result['absent_classes'])
print('Classes:', report['classes'])
print('Download item_classifier.pt and classifier_report.json from /kaggle/working')
